In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import pandas as pd
from datetime import datetime as dt
from rockyelevate.wrapper import Session
from rockyelevate.utils import response_to_dataframe
from rockyclickup.wrapper import Session as cu
from rockyclickup.models import Client
from rockyclickup.database_interface import get_task
from rockyclickup.utils import response_to_dataframe as clickup_response_to_dataframe

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from constants.maps import (
    ELV_ACCOUNT_TYPE_MAP,
    ORGANIZATION_RENAME_MAP,
    ELEVATE_PLAN_RENAME_MAP,
    CLIENT_RENAME_MAP,
    CLICKUP_PLAN_RENAME_MAP,
    COLS_TO_DROP,
)

from utils.clickup import fix_clickup_date

# from engine.apps.meridian.constants import (
#     ORGANIZATION_RENAME_MAP,
#     ELEVATE_PLAN_RENAME_MAP,
#     CLIENT_RENAME_MAP,
#     CLICKUP_PLAN_RENAME_MAP,
#     COLS_TO_DROP,
# )

In [3]:
clickup = cu()

elevate_server = "PROD"
elv = Session(elevate_server, multithread=True, max_threads=40)
today = dt.now()

In [4]:
''' Get all Elevate Organizations ''' # ~11m
try:
    all_orgs_df = pd.read_pickle(f"cache/{dt.strftime(today, "%y%m%d")}_ALL_ORGS_{elevate_server}.pkl")
    print(f"opened {len(all_orgs_df)} organizations")

except FileNotFoundError:
    # 
    print(f"fetching organizations from elevate, please wait...")
    all_orgs_res = elv.get_organizations(
        statuses=["PENDING", "ACTIVE", "TERMINATED", "ACTIVATION_FAILED"],
        details=["STATUS"],
        types=["SYSTEM", "PARTNER", "DISTRIBUTOR", "COMPANY", "SUBSIDIARY", "SUBGROUP"],
        subsidiaries="include"
    )
    all_orgs_df = response_to_dataframe(all_orgs_res)
    all_orgs_df = all_orgs_df.rename(columns=ORGANIZATION_RENAME_MAP).drop(columns=COLS_TO_DROP, errors='ignore')
    all_orgs_df.to_pickle(f"cache/{dt.strftime(today, "%y%m%d")}_ALL_ORGS_{elevate_server}.pkl")

    print(f"found {len(all_orgs_df)} organizations")

fetching organizations from elevate, please wait...
[range(0, 2), range(2, 4), range(4, 6), range(6, 8), range(8, 10), range(10, 12), range(12, 14), range(14, 16), range(16, 18), range(18, 20), range(20, 22), range(22, 24), range(24, 26), range(26, 28), range(28, 30), range(30, 32), range(32, 34), range(34, 36), range(36, 38), range(38, 40), range(40, 42), range(42, 44), range(44, 46), range(46, 48), range(48, 50), range(50, 52), range(52, 54), range(54, 56), range(56, 58), range(58, 60), range(60, 62), range(62, 64), range(64, 66), range(66, 68), range(68, 70), range(70, 72), range(72, 74), range(74, 76), range(76, 78), range(78, 80)]
1591
pool
[16421, 16422, 16454, 24780, 24781, 24782, 24783, 24784, 24802, 8441, 8487, 8488, 8489, 8491, 8492, 8493, 8494, 8495, 8496, 8497, 8498, 8499, 8500, 8501, 8502, 8503, 8504, 8505, 8506, 8507, 8508, 8509, 8510, 8511, 8512, 8513, 8514, 8515, 8516, 8517, 8518, 8519, 8520, 8521, 8522, 8523, 8524, 8525, 8526, 8527, 8528, 8529, 8530, 8531, 8532, 8533, 

In [5]:
''' Get all Elevate Plans ''' # ~2m
elv_plans_cache_path = f"cache/{dt.strftime(today, "%y%m%d")}_ALL_PLANS_{elevate_server}.pkl"
try:
    elv_plans_df = pd.read_pickle(elv_plans_cache_path)
    print(f"opened {len(elv_plans_df)} elevate plans")

except FileNotFoundError:
    print(f"fetching plans from elevate, please wait...")
    elv_plans_res = elv.get_plans_by_org(oids=all_orgs_df['organization_id'].to_list(), detail=True)
    elv_plans_df = response_to_dataframe(elv_plans_res)
    elv_plans_df = elv_plans_df.rename(columns=ELEVATE_PLAN_RENAME_MAP).drop(columns=COLS_TO_DROP, errors='ignore')
    
    for col in ['plan_year.valid_from', 'plan_year.valid_to']:
        if col not in elv_plans_df:
            print(f"{col} not in plan df")
            continue
        elv_plans_df[col] = pd.to_datetime(elv_plans_df[col])
    elv_plans_df.to_pickle(elv_plans_cache_path)

    print(f"found {len(elv_plans_df)} elevate plans")

fetching plans from elevate, please wait...
found 5846 elevate plans


In [6]:
''' Get all ClickUp Clients ''' # ~10s
clients_cache_path = f"cache/{dt.strftime(today, "%y%m%d")}_ALL_CLIENTS_CLICKUP.pkl"

try:
    all_clients_df = pd.read_pickle(clients_cache_path)
    print(f"opened {len(all_clients_df)} clickup clients")

except FileNotFoundError:
    print(f"fetching clients from clickup, please wait...")
    all_clients_res = clickup.get_full_list(model=Client)
    all_clients_df = clickup_response_to_dataframe(all_clients_res)
    all_clients_df = all_clients_df.rename(columns=CLIENT_RENAME_MAP).drop(columns=COLS_TO_DROP + ['list_id', 'task_type'], errors='ignore')
    all_clients_df.to_pickle(clients_cache_path)
    print(f"found {len(all_clients_df)} clickup clients")

fetching clients from clickup, please wait...
Field not in config.db: waterfall 702d85f6-7155-447c-9019-206db17ab27c
Field not in config.db: ducks 80f59460-7b7d-4652-9d9c-f1029b146ede
Field not in config.db: data_transmission_details 2bd919bf-6695-4f13-9f80-9e7b077bc83c
Field not in config.db: divisional_invoicing 951e9aa5-c6d9-4ad1-98a9-118b5d478640
Field not in config.db: temp_am_cobra f1c08a6f-3a1c-4392-8967-bbcd9501e1a8
Field not in config.db: temp_account_manager 159d31bb-e617-4d3a-b900-021cc0ad4542
Field not in config.db: data_start 7d44c3ea-fbe2-42d7-a21a-0e8f42bd8f1b
Field not in config.db: temp_flex_divisions 0d2211c9-b8c9-417e-b020-74ca30273509
Field not in config.db: temp_cobra_divisions e1798c36-372a-446b-8612-de839429d43c
Field not in config.db: data_end 3b8c9ecd-1e23-442a-8155-2b15c1559dee
found 2732 clickup clients


In [7]:
''' Get all ClickUp Plans ''' # ~30s
cu_plans_cache_path = f"cache/{dt.strftime(today, '%y%m%d')}_ALL_PLANS_CLICKUP.pkl"

collected_plan_types = []

try:
    clickup_plans_df = pd.read_pickle(cu_plans_cache_path)
    print(f"opened {len(clickup_plans_df)} clickup plans")

except FileNotFoundError:
    rel_account_types = elv_plans_df['account_type.account_type'].unique()

    all_clickup_plans = []
    for eat in rel_account_types:
        try:
            norm_type = ELV_ACCOUNT_TYPE_MAP.get(eat, None).lower()
            if norm_type in collected_plan_types:
                continue
            collected_plan_types.append(norm_type)

            if not norm_type:
                print(f"Warning: No mapping found for account type `{eat}`")
                continue

            rcu_db_plan = get_task(category=norm_type)
            if not rcu_db_plan:
                print(f"Warning: No database plan found for '{norm_type}'")
                continue

            list_id = rcu_db_plan.list_id

            list_plans = clickup.get_full_list(list_id=list_id)
            if list_plans:
                all_clickup_plans.extend(list_plans)           
        
        except Exception as e:
            print(f"Error fetching {norm_type} plans: {e}")

    clickup_plans_df = clickup_response_to_dataframe(all_clickup_plans)
    clickup_plans_df = clickup_plans_df.rename(columns=CLICKUP_PLAN_RENAME_MAP)

    for col in ['date_plan_start', 'date_plan_end']:
        if col in clickup_plans_df.columns:
            clickup_plans_df[col] = clickup_plans_df[col].apply(
                lambda x: fix_clickup_date(x) if x is not None else None
            )
            clickup_plans_df[col] = clickup_plans_df[col].apply(
                lambda x: x.replace(tzinfo=None) if x is not None and hasattr(x, 'tzinfo') else x
            )

    clickup_plans_df.to_pickle(cu_plans_cache_path)
    print(f"found {len(clickup_plans_df)} clickup plans")

found 6036 clickup plans


In [8]:
all_orgs_df_COPY = all_orgs_df.copy()
elv_plans_df_COPY = elv_plans_df.copy()
all_clients_df_COPY = all_clients_df.copy()
clickup_plans_df_COPY = clickup_plans_df.copy()

In [9]:
''' rename columns '''

all_orgs_df = all_orgs_df_COPY.rename(columns=ORGANIZATION_RENAME_MAP)
elv_plans_df = elv_plans_df_COPY.rename(columns=ELEVATE_PLAN_RENAME_MAP)
all_clients_df = all_clients_df_COPY.rename(columns=CLIENT_RENAME_MAP)
clickup_plans_df = clickup_plans_df_COPY.rename(columns=CLICKUP_PLAN_RENAME_MAP)


In [10]:
all_clients_df["elevate_auto_renew"].value_counts()


elevate_auto_renew
Active    1772
Paused       2
Name: count, dtype: int64

In [ ]:
''' Merge all dataframes together ''' # ~11s
merge_cache_path = f"cache/{dt.strftime(today, '%y%m%d')}_ALL_PLANS_ORGS_CLIENTS.pkl"

elv_plans_orgs_df = pd.merge(left=elv_plans_df, right=all_orgs_df, on=['organization_id'], how='left')
elv_plans_orgs_clients_df = pd.merge(left=elv_plans_orgs_df, right=all_clients_df, on='rmrcode', how='left')

def extract_client_ids(row):
    client_ids = []
    for field_value in row:
        if isinstance(field_value, list):
            client_ids.extend(field_value)
        elif field_value is not None:
            client_ids.append(field_value)
    print(client_ids)
    return client_ids

def assign_client_id(filtered_list):
    filtered_list = [c for c in filtered_list if c in all_clients_df['client_id'].unique()]
    
    unique = list(set(filtered_list))
    
    if len(unique) == 0:
        return None
    
    if len(unique) == 1:
        return unique[0]

    return unique[0]

def match_clickup_plan(row):
    mapped_account_type = ELV_ACCOUNT_TYPE_MAP.get(row['account_type.account_type'])

    client_match = clickup_plans_df['client_id'] == row['client_id']
    type_match = clickup_plans_df['cu_account_type'] == mapped_account_type.upper()
    date_match = clickup_plans_df['date_plan_start'] == row['plan_year.valid_from']

    plan_search = clickup_plans_df[client_match & type_match & date_match]

    plan_id_search = list(set(plan_search['cu_plan_id'].to_list()))

    if not plan_id_search:
        return None

    if len(plan_id_search) == 1:
        return plan_id_search[0]
    
    return None

relation_fields = [c for c in clickup_plans_df.columns if 'client' in c]
clickup_plans_df[relation_fields] = clickup_plans_df[relation_fields].apply(
    lambda col: col.apply(lambda x: x if isinstance(x, list) else [])
)
clickup_plans_df['client_id_list'] = clickup_plans_df.apply(lambda row: sum(row[relation_fields].values, []), axis=1)
clickup_plans_df['client_id_list'] = clickup_plans_df['client_id_list'].apply(lambda x: list(set(x)))
clickup_plans_df['client_id'] = clickup_plans_df['client_id_list'].apply(assign_client_id)

elv_plans_orgs_clients_df['cu_plan_id'] = elv_plans_orgs_clients_df.apply(match_clickup_plan, axis=1)
clickup_plans_df = clickup_plans_df.drop(columns=['client_id'], errors='ignore')
merge_df = pd.merge(left=elv_plans_orgs_clients_df, right=clickup_plans_df, on='cu_plan_id', how='left')

merge_df.to_pickle(merge_cache_path)

print(f"{len(merge_df)} total plan rows")
print(f"-"*60)
''' rows that don't have clickup client id '''
missing_client = merge_df[pd.isna(merge_df['client_id'])]
print(f"{len(missing_client):>5} ({len(missing_client)/len(merge_df):>3.0%}) rows do NOT have a ClickUp Client ID")

''' rows that don't have matching clickup plan id '''
missing_cu_plan = merge_df[pd.isna(merge_df['cu_plan_id'])]
print(f"{len(missing_cu_plan):>5} ({len(missing_cu_plan)/len(merge_df):>3.0%}) rows do NOT have a ClickUp Plan ID")

5848 total plan rows
------------------------------------------------------------
    9 ( 0%) rows do NOT have a ClickUp Client ID
 1187 (20%) rows do NOT have a ClickUp Plan ID


In [12]:
cu_plans_wo_elevate = clickup_plans_df[~clickup_plans_df['cu_plan_id'].isin(merge_df['cu_plan_id'])]
print(len(cu_plans_wo_elevate))

1401


In [13]:
missing_cu_plan['account_type.account_type'].value_counts()

account_type.account_type
HSA          932
DCAP          86
HCFSA         83
HRA           47
TRANSIT       13
LIFESTYLE     13
PARKING        9
SPECIALTY      2
ADOPTION       2
Name: count, dtype: int64

In [14]:
print(len(clickup_plans_df))

6036


In [ ]:
# 4646/5931 elv plans have a matching clickup card (78.3%)
# 4622/6036  cu cards have a matching elevate plan (76.6%)